# Lesson 5 — Three Tools Every Scientist Needs: Fitting, Statistics, Automation

This lesson contains **three independent subchapters**. Each one re-imports what it needs and loads its own data, so after the short setup cell below you can run them **in any order**.

## The three tools

1. **Fitting** — describe experimental data with a mathematical model and estimate its parameters (with their uncertainty).
2. **Statistics and uncertainty** — decide whether an observed difference is meaningful or just compatible with noise.
3. **Automation** — repeat the same analysis over many files instead of doing it by hand, one by one.

## Data

We reuse the same tensile and hardness datasets as the previous lessons (`tensile_raw.csv`, `hardness_positions.csv`). Part C builds a small `runs/` folder from them on the fly.

## Important note

The goal is **not** to become experts in constitutive modelling or statistical inference today. The goal is to see three practical ideas:

```text
fit a model  ->  quantify uncertainty  ->  automate repetitive analysis
```

In [ ]:
# Setup cell: run this once at the beginning.
# It installs SciPy (used in Parts A and B). This is plumbing — just run it.
%pip install -q scipy
print("Setup complete. You can now run Part A, B or C in any order.")

# Part A — Nonlinear fitting with SciPy

In Lesson 4 we fitted a **straight line** to the initial elastic region of a stress-strain curve.

But a real stress-strain curve is not straight: after the elastic part it bends over as the material hardens, reaches a maximum (the **ultimate tensile strength**, UTS), and then falls as the specimen necks.

Here we model the **hardening region, up to the UTS**, of specimen **S03** with a simple Voce-type saturation law:

$$
\sigma(\epsilon) = \sigma_s - (\sigma_s - \sigma_0)\,\exp\!\left(-\frac{\epsilon}{\epsilon_c}\right)
$$

- $\sigma_s$ — saturation stress (the plateau the curve heads toward);
- $\sigma_0$ — model stress at zero strain;
- $\epsilon_c$ — how fast the curve approaches saturation.

In [ ]:
# Part A — imports and data loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def load_table(name):
    """Robust CSV loader: works locally and in the browser (JupyterLite).
    Do not worry about the details — this is just plumbing."""
    for folder in ("data", "../data", "."):
        try:
            return pd.read_csv(f"{folder}/{name}")
        except Exception:
            pass
    import js
    from pyodide.http import open_url
    base = str(js.location.href).split("/extensions/")[0]
    return pd.read_csv(open_url(f"{base}/files/data/{name}"))

tensile = load_table("tensile_raw.csv")
print(tensile.shape)
tensile.head()

In [ ]:
# Engineering stress and strain from the raw quantities
tensile["area_mm2"] = np.pi * (tensile["diameter_mm"] / 2) ** 2
tensile["stress_MPa"] = tensile["force_N"] / tensile["area_mm2"]
tensile["strain"] = tensile["displacement_mm"] / tensile["gauge_length_mm"]

# Pick one specimen and keep only the hardening region, up to the peak (UTS)
sample_id = "S03"
sample = tensile[tensile["sample_id"] == sample_id].copy().reset_index(drop=True)

peak_idx = sample["stress_MPa"].idxmax()          # row of the ultimate tensile strength
hardening = sample.loc[:peak_idx]                 # everything up to and including the peak

print(f"{sample_id}: {len(sample)} points total, peak at point {peak_idx} "
      f"(UTS = {sample.loc[peak_idx, 'stress_MPa']:.0f} MPa)")
print(f"Fitting the first {len(hardening)} points (up to the UTS).")

In [ ]:
# Visual inspection: full curve, with the region we will fit highlighted
plt.figure(figsize=(6, 4))
plt.plot(sample["strain"], sample["stress_MPa"], "o", markersize=3, color="0.7", label="full curve")
plt.plot(hardening["strain"], hardening["stress_MPa"], "o", markersize=4, label="fit region (up to UTS)")
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title(f"Stress-strain curve: {sample_id}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Define the Voce-type model as a Python function
def voce_model(strain, sigma_s, sigma_0, epsilon_c):
    return sigma_s - (sigma_s - sigma_0) * np.exp(-strain / epsilon_c)

# curve_fit needs a starting guess p0 for a nonlinear model.
# Rough reading off the plot above: saturates near ~500 MPa, starts near 0, knee around 0.01.
p0 = [500, 10, 0.01]
print("Initial guesses -> sigma_s =", p0[0], "MPa, sigma_0 =", p0[1], "MPa, epsilon_c =", p0[2])

In [ ]:
# Run the nonlinear fit on the hardening region
x = hardening["strain"].to_numpy()
y = hardening["stress_MPa"].to_numpy()

popt, pcov = curve_fit(voce_model, x, y, p0=p0, maxfev=10000)
sigma_s, sigma_0, epsilon_c = popt

# Parameter uncertainties: square roots of the diagonal of the covariance matrix
sigma_s_err, sigma_0_err, epsilon_c_err = np.sqrt(np.diag(pcov))

# Goodness of fit: coefficient of determination R^2
y_fit = voce_model(x, *popt)
residuals = y - y_fit
r_squared = 1 - np.sum(residuals**2) / np.sum((y - np.mean(y))**2)

print("Fitted parameters (value +/- uncertainty):")
print(f"  sigma_s   = {sigma_s:7.2f} +/- {sigma_s_err:.2f} MPa")
print(f"  sigma_0   = {sigma_0:7.2f} +/- {sigma_0_err:.2f} MPa")
print(f"  epsilon_c = {epsilon_c:7.4f} +/- {epsilon_c_err:.4f}")
print(f"  R^2       = {r_squared:.4f}")

In [ ]:
# Plot the data and the fitted curve
x_smooth = np.linspace(x.min(), x.max(), 300)

plt.figure(figsize=(6, 4))
plt.plot(sample["strain"], sample["stress_MPa"], "o", markersize=3, color="0.7", label="full curve")
plt.plot(x, y, "o", markersize=4, label="fit region")
plt.plot(x_smooth, voce_model(x_smooth, *popt), "-", linewidth=2, color="red", label="Voce fit")
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title(f"Nonlinear fit of the hardening region: {sample_id}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Estimated saturation stress: sigma_s = {sigma_s:.0f} +/- {sigma_s_err:.0f} MPa")

## Optional: residuals

Residuals are the differences between measured and fitted values:

```text
residual = data - fit
```

If the model is appropriate, residuals scatter around zero without a strong systematic pattern.

In [ ]:
# Optional residual plot
plt.figure(figsize=(6, 3.5))
plt.axhline(0, linestyle="--", linewidth=1, color="0.4")
plt.plot(x, residuals, "o", markersize=4)
plt.xlabel("Strain [-]")
plt.ylabel("Residual [MPa]")
plt.title(f"Residuals of the Voce fit: {sample_id}")
plt.grid(True, alpha=0.3)
plt.show()

## Exercise A — Fit another specimen

Repeat the fit for another specimen, e.g. `S01` or `S04`.

1. Is the fitted saturation stress higher or lower than for `S03`?
2. Is the fit quality (R^2) similar?
3. Do the residuals show a systematic pattern?

In [ ]:
# Exercise A stub
# 1. Choose another sample ID
other_id = "S01"

# 2. Select its rows and keep the hardening region up to the peak
# other = tensile[tensile["sample_id"] == other_id].copy().reset_index(drop=True)
# other_peak = other["stress_MPa"].idxmax()
# other_hardening = other.loc[:other_peak]

# 3. Build x and y, run curve_fit with the same p0
# 4. Print sigma_s +/- its uncertainty and R^2, then plot data + fit
# YOUR CODE HERE

# Part B — Statistics and uncertainty

A very common scientific question:

> The quenched samples *look* harder than the annealed samples. Is the difference **meaningful**, or could it just be noise?

Using repeated hardness measurements, we introduce: mean, standard deviation, standard error of the mean, error bars, and a two-sample t-test.

In [ ]:
# Part B — imports and data loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

def load_table(name):
    """Robust CSV loader: works locally and in the browser (JupyterLite).
    Do not worry about the details — this is just plumbing."""
    for folder in ("data", "../data", "."):
        try:
            return pd.read_csv(f"{folder}/{name}")
        except Exception:
            pass
    import js
    from pyodide.http import open_url
    base = str(js.location.href).split("/extensions/")[0]
    return pd.read_csv(open_url(f"{base}/files/data/{name}"))

hardness = load_table("hardness_positions.csv")

# Keep only the measurements flagged as valid (same data-quality step as Lesson 4)
hardness = hardness[hardness["is_valid"]]
print(hardness.shape)
hardness.head()

## Standard deviation vs standard error of the mean

The **standard deviation** describes the spread of the individual measurements.

The **standard error of the mean** describes the uncertainty on the *estimated mean*:

$$
SEM = \frac{\text{std}}{\sqrt{n}}
$$

They answer different questions:

```text
std  ->  how scattered are the individual measurements?
SEM  ->  how uncertain is the mean we computed?
```

In [ ]:
# Summary statistics by treatment
summary = (
    hardness
    .groupby("treatment")["hardness_HV"]
    .agg(mean="mean", std="std", n="count")
    .reset_index()
)
summary["sem"] = summary["std"] / np.sqrt(summary["n"])
summary

In [ ]:
# Bar chart: mean hardness with mean +/- SEM error bars
plt.figure(figsize=(6, 4))
plt.bar(summary["treatment"], summary["mean"], yerr=summary["sem"], capsize=5)
plt.ylabel("Hardness [HV]")
plt.xlabel("Treatment")
plt.title("Hardness by treatment: mean +/- SEM")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# Two-sample t-test: is quenched really harder than annealed?
annealed = hardness.loc[hardness["treatment"] == "annealed", "hardness_HV"]
quenched = hardness.loc[hardness["treatment"] == "quenched", "hardness_HV"]

# Welch's t-test: does not assume the two groups have equal variance
result = stats.ttest_ind(quenched, annealed, equal_var=False)

print("Two-sample t-test: quenched vs annealed")
print(f"  mean quenched = {quenched.mean():.1f} HV   mean annealed = {annealed.mean():.1f} HV")
print(f"  t statistic   = {result.statistic:.3f}")
print(f"  p-value       = {result.pvalue:.3e}")

if result.pvalue < 0.05:
    print("  Conclusion: the difference is statistically significant at the 0.05 level.")
else:
    print("  Conclusion: not enough evidence for a difference at the 0.05 level.")

## Exercise B — Compare another pair of treatments

Compare, for example, `aged` vs `as_received`, or `quenched` vs `aged`.

1. Which treatment has the higher mean hardness?
2. What is the p-value?
3. What single sentence would you write as a conclusion?

In [ ]:
# Exercise B stub
# t1, t2 = "aged", "as_received"
# values_1 = hardness.loc[hardness["treatment"] == t1, "hardness_HV"]
# values_2 = hardness.loc[hardness["treatment"] == t2, "hardness_HV"]
# result = stats.ttest_ind(values_1, values_2, equal_var=False)
# print(result.pvalue)
# YOUR CODE HERE

# Part C — Automation: many files

So far we worked with one table at a time. In real research you often have **many** files — one per test:

```text
runs/S01.csv
runs/S02.csv
runs/S03.csv
...
```

Doing the same analysis by hand on each file is slow and error-prone. This is where Python clearly beats a manual spreadsheet workflow: write the analysis **once**, run it on **every** file.

In [ ]:
# Part C — prepare the example files, then discover them
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import glob

def load_table(name):
    """Robust CSV loader: works locally and in the browser (JupyterLite).
    Do not worry about the details — this is just plumbing."""
    for folder in ("data", "../data", "."):
        try:
            return pd.read_csv(f"{folder}/{name}")
        except Exception:
            pass
    import js
    from pyodide.http import open_url
    base = str(js.location.href).split("/extensions/")[0]
    return pd.read_csv(open_url(f"{base}/files/data/{name}"))

# In real research these per-specimen files would already exist (one per test).
# Here we create them from the master table so the example is self-contained.
RUNS_DIR = Path("runs")
RUNS_DIR.mkdir(exist_ok=True)
master = load_table("tensile_raw.csv")
for sid, group in master.groupby("sample_id"):
    group.to_csv(RUNS_DIR / f"{sid}.csv", index=False)

# glob finds all files matching a pattern in a folder
files = sorted(glob.glob(str(RUNS_DIR / "*.csv")))
print(f"Found {len(files)} files:")
for f in files:
    print("  ", f)

In [ ]:
# Define a function that analyses ONE tensile-test file
def analyse_tensile_file(file_path):
    df = pd.read_csv(file_path)
    df["area_mm2"] = np.pi * (df["diameter_mm"] / 2) ** 2
    df["stress_MPa"] = df["force_N"] / df["area_mm2"]
    df["strain"] = df["displacement_mm"] / df["gauge_length_mm"]

    peak = df["stress_MPa"].idxmax()
    return {
        "sample_id": df["sample_id"].iloc[0],
        "treatment": df["treatment"].iloc[0],
        "UTS_MPa": df.loc[peak, "stress_MPa"],
        "strain_at_UTS": df.loc[peak, "strain"],
        "n_points": len(df),
    }

In [ ]:
# Loop over all files and collect one row per specimen
records = []
for f in files:
    records.append(analyse_tensile_file(f))

all_results = pd.DataFrame(records)
all_results

In [ ]:
# Save the automated summary table
output_path = Path("automated_tensile_summary.csv")
all_results.to_csv(output_path, index=False)
print("Saved:", output_path.resolve())

In [ ]:
# Plot every specimen's curve with a single loop
plt.figure(figsize=(7, 4))
for f in files:
    df = pd.read_csv(f)
    df["stress_MPa"] = df["force_N"] / (np.pi * (df["diameter_mm"] / 2) ** 2)
    df["strain"] = df["displacement_mm"] / df["gauge_length_mm"]
    plt.plot(df["strain"], df["stress_MPa"], label=df["sample_id"].iloc[0])

plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title("All tensile-test files, plotted automatically")
plt.legend(ncol=2)
plt.grid(True, alpha=0.3)
plt.show()

## Exercise C — Add one column to the pipeline

Modify `analyse_tensile_file()` so the output table also contains one extra quantity, for example:

- final strain or final stress;
- mean stress;
- the file name.

Then re-run the loop and save the updated table.

In [ ]:
# Exercise C stub
# 1. Copy analyse_tensile_file() here and add one new key to the returned dict.
# 2. Re-run the loop over `files` to rebuild all_results.
# 3. Save the updated table with to_csv.
# YOUR CODE HERE

# Closing — Course recap

This final lesson ties together the whole course:

```text
L1  ->  Foundations & first look
        variables, types, lists, conditions, loops, functions; a read->compute->plot demo

L2  ->  Python basics (hands-on)
        arithmetic, booleans, if/elif, lists, for, functions, dictionaries, basic plots

L3  ->  From raw files to clean data tables
        read_csv, inspect, filter, calculated columns, cleaning, groupby, save

L4  ->  Plotting and data analysis
        line/scatter/hist/box/bar plots, summary metrics, a linear fit

L5  ->  Three practical tools
        nonlinear fitting, uncertainty & statistics, automation over many files
```

The central message:

```text
Python pays off when the analysis must be repeated, checked, modified, or shared.
```

# Where to go next

Three useful directions after this introductory course:

## 1. SciPy
Fitting, optimization, statistics, interpolation, integration, signal processing.
Start with: `scipy.optimize.curve_fit`, `scipy.stats`, interpolation.

## 2. pandas
Tabular data analysis.
Start with: missing values, merging tables, grouping/aggregation, reshaping.

## 3. Matplotlib
Publication-quality figures.
Start with: `fig, ax = plt.subplots()`, axis customization, saving vector graphics, multi-panel figures.

Final practical advice:

```text
Start with small notebooks that solve real problems from your own research.
```